# Spatial Intelligence - Part 3

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)


## 5. Import the gallery floor plan

In [ ]:
gallery = Topology.ByBREPPath(r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\MachineLearning\gallery.brep")

## 6. Show the geometry

In [ ]:
Topology.Show(gallery,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Create a grid overlay

In [ ]:
b_r = Wire.BoundingRectangle(gallery)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange1 = list(range(0,int(width)+28,28))
vRange1 = list(range(0,int(length)+28,28))

uRange2 = list(range(0,int(width)+3,3))
vRange2 = list(range(0,int(length)+3,3))
grid1 = Grid.VerticesByDistances(gallery, clip=True, uRange=uRange1, vRange=vRange1)
grid2 = Grid.EdgesByDistances(gallery, clip=True, uRange=uRange2, vRange=vRange2)

## 8. Slice the floor plan with the edge grid to create a topologic shell

In [ ]:
shell = Topology.Slice(gallery, grid2)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## 9. Derive analysis graphs from the shell

In [ ]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
analysis_graph = Graph.ByTopology(shell)

## 10. Derive and store the graph vertices and isovist vertices

In [ ]:
g_verts = Graph.Vertices(analysis_graph)
iso_verts = Topology.Vertices(grid1)

## 11. Spatial Intelligence through Isovists and Graph Analysis

## Visibility Graph Analysis
### Create Isovists
* Time consuming (about 20 minutes!)
* Will print out errors. Ignore.

In [ ]:
isovists = []
for v in iso_verts:
    isovist = Face.Isovist(gallery, v)
    isovists.append(isovist)

In [ ]:
Topology.Show(gallery, isovists,
              faceColorKey="cp_color",
              faceOpacity=0.6,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### Compute the visibility of each isovist viewpoint
* Calculate how many other points of the dense grid are within each isovist's face.

In [ ]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso: # Skip is iso is None
        v = iso_verts[i]
        b_list = Vertex.IsInternal2D(g_verts, iso)
        b_list = [b for b in b_list if b]
        n = len(b_list)
        n_list.append(n)
        d = Dictionary.ByKeyValue("visibility", n)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)



### Transfer/Interpolate values from the new graph vertices to the original graph vertices

In [ ]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

### Derive the colour of each vertex based on the interpolated value

In [ ]:
minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

### Transfer the information from the graph vertices to the faces of the original shell

In [ ]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)